In [61]:
# Download data dan src dari GitHub (uncomment jika di Colab)
!rm -rf IR-BERT-Transformer data src
!git clone https://github.com/teranixbq/IR-BERT-Transformer.git
!mv IR-BERT-Transformer/data ./data
!mv IR-BERT-Transformer/src ./src
!rm -rf IR-BERT-Transformer

Cloning into 'IR-BERT-Transformer'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 76 (delta 33), reused 54 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 38.90 KiB | 12.97 MiB/s, done.
Resolving deltas: 100% (33/33), done.


# Neural IR Exercise dengan BERT Cross-Encoder
Dosen: Zico Pratama Putra
Kelompok: [Nama 1] - [Nama 2] - [Nama 3]
Tanggal: ---

# **A. Setup**

In [62]:
from src.judgement_aggregation import load_raw_judgements, aggregate_judgements, save_qrels, compute_annotator_reliability
from src.bert_cross_encoder import BERTCrossEncoder
from transformers import pipeline
import pandas as pd

DEST = "/content/data/"
RAW_JUDGEMENTS = DEST + "fira_raw_judgements.tsv"
BASELINE_2022 = DEST + "fira-2022.baseline-qrels.tsv"

# **B. Load Raw**

In [63]:
df = load_raw_judgements(RAW_JUDGEMENTS)

Loaded 237 raw judgements
Columns: ['query_id', 'doc_id', 'judgement', 'confidence', 'annotator_id', 'duration_ms']
   query_id doc_id  judgement  confidence annotator_id  duration_ms
0         1   d1_1          3        0.85       User_5        30289
1         1   d1_1          3        0.94       User_0        83482
2         1   d1_1          3        0.92       User_4        16165
3         1   d1_2          2        0.81       User_0        38062
4         1   d1_2          1        0.92       User_5        12851


# **C. Helper Function**

In [64]:
def compare_with_baseline(
    aggregated_df: pd.DataFrame,
    baseline_qrels: pd.DataFrame
):
    """
    Membandingkan advanced aggregation dengan baseline qrels.

    Metrik:
    - matched query-doc pairs
    - exact match
    - exact match rate
    - mean absolute difference
    """

    merged = aggregated_df.merge(
        baseline_qrels[["query_id", "doc_id", "baseline_score"]],
        on=["query_id", "doc_id"],
        how="inner"
    )

    merged["score_diff"] = (
        merged["score"] -
        merged["baseline_score"]
    )

    merged["abs_diff"] = merged["score_diff"].abs()

    total_pairs = len(merged)
    exact_match = (merged["score_diff"] == 0).sum()

    exact_match_rate = (
        exact_match / total_pairs
        if total_pairs > 0
        else 0
    )

    mean_abs_diff = (
        merged["abs_diff"].mean()
        if total_pairs > 0
        else 0
    )

    print("=== Comparison with Baseline Qrels ===")
    print(f"Matched query-doc pairs  : {total_pairs}")
    print(f"Exact match              : {exact_match}")
    print(f"Exact match rate         : {exact_match_rate:.4f}")
    print(f"Mean absolute difference : {mean_abs_diff:.4f}")

    print("\nDistribusi perbedaan score:")
    print(merged["score_diff"].value_counts().sort_index())

    return merged

# **PART 1 : FiRA Judgement Aggregation**

## 1.1 Baseline - Simple Majority Vote

In [65]:
agg_maj = aggregate_judgements(df, method='majority')
agg_maj.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.000000
1,1,d1_2,2,3,0.471405
2,1,d1_3,1,3,0.471405
3,1,d1_4,0,3,0.000000
4,1,d1_5,0,3,0.000000
5,1,d1_6,0,3,0.000000
6,1,d1_7,1,3,0.000000
7,1,d1_8,1,3,0.471405
8,2,d2_1,3,3,0.471405
9,2,d2_2,2,3,0.471405


## 1.2 Confidence-Based Weighting
Bobot berdasarkan seberapa yakin annotator

In [66]:
agg_weighted = aggregate_judgements(df, method='advanced', strategy='confidence_weighted')
agg_weighted.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.000000
1,1,d1_2,2,3,0.471405
2,1,d1_3,1,3,0.471405
3,1,d1_4,0,3,0.000000
4,1,d1_5,0,3,0.000000
5,1,d1_6,0,3,0.000000
6,1,d1_7,1,3,0.000000
7,1,d1_8,1,3,0.471405
8,2,d2_1,3,3,0.471405
9,2,d2_2,2,3,0.471405


## 1.3 Reliability-Weighted Voting
Bobot dari track record annotator — seberapa sering ia setuju dengan suara terbanyak (majority vote). Annotator yang sering benar dapat bobot lebih besar.

In [67]:
reliability = compute_annotator_reliability(df)
reliability

{'User_0': 0.7692307692307693,
 'User_1': 0.8372093023255814,
 'User_2': 0.7804878048780488,
 'User_3': 0.7659574468085106,
 'User_4': 0.8275862068965517,
 'User_5': 0.7631578947368421}

In [68]:
agg_reliability = aggregate_judgements(df, method='advanced', strategy='annotator_reliability', annotator_reliability=reliability)
agg_reliability.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,std_score
0,1,d1_1,3,3,0.000000
1,1,d1_2,2,3,0.471405
2,1,d1_3,1,3,0.471405
3,1,d1_4,0,3,0.000000
4,1,d1_5,0,3,0.000000
5,1,d1_6,0,3,0.000000
6,1,d1_7,1,3,0.000000
7,1,d1_8,1,3,0.471405
8,2,d2_1,3,3,0.471405
9,2,d2_2,2,3,0.471405


## 1.4 Bandingkan Majority Dengan Advanced

### 1.4.1 Majority & Advanced (Confidence-Based Weighting)

In [69]:
comparison_df1 = agg_maj.merge(
    agg_weighted,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df1["score_diff"] = (
    comparison_df1["score_advanced"] -
    comparison_df1["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df1))
print("Jumlah score berbeda :", (comparison_df1["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df1["score_diff"].value_counts().sort_index())

display(comparison_df1.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,std_score_majority,score_advanced,num_judgements_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,0.000000,3,3,0.000000,0
1,1,d1_2,2,3,0.471405,2,3,0.471405,0
2,1,d1_3,1,3,0.471405,1,3,0.471405,0
3,1,d1_4,0,3,0.000000,0,3,0.000000,0
4,1,d1_5,0,3,0.000000,0,3,0.000000,0


### 1.4.2 Majority & Advanced (Reliability-Weighted Voting)

In [70]:
comparison_df2 = agg_maj.merge(
    agg_reliability,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df2["score_diff"] = (
    comparison_df2["score_advanced"] -
    comparison_df2["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df2))
print("Jumlah score berbeda :", (comparison_df2["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df2["score_diff"].value_counts().sort_index())

display(comparison_df2.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,std_score_majority,score_advanced,num_judgements_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,0.000000,3,3,0.000000,0
1,1,d1_2,2,3,0.471405,2,3,0.471405,0
2,1,d1_3,1,3,0.471405,1,3,0.471405,0
3,1,d1_4,0,3,0.000000,0,3,0.000000,0
4,1,d1_5,0,3,0.000000,0,3,0.000000,0


## 1.5 Bandingkan BASELINE dengan Advanced

In [71]:
baseline_df = pd.read_csv(BASELINE_2022, sep=r"\s+", names=["query_id", "unused", "doc_id", "baseline_score"])

### 1.5.1 BASELINE & Advanced (Confidence - Based Weighting)

In [72]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_weighted,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,0.000000,3,0,0
1,1,d1_2,2,3,0.471405,2,0,0
2,1,d1_3,1,3,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0,0,0


### 1.5.2 BASELINE & Advanced (Reliability-Weighted Voting)

In [73]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_reliability,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,0.000000,3,0,0
1,1,d1_2,2,3,0.471405,2,0,0
2,1,d1_3,1,3,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0,0,0


## 1.7 Analisis Manual Disagreement Tinggi

In [78]:
def inspect_high_disagreement_cases(
    raw_df: pd.DataFrame,
    aggregated_df: pd.DataFrame,
    top_n=5
):
    """
    Menampilkan contoh query-doc pair dengan disagreement tertinggi.

    Tujuan:
    - memeriksa variasi judgement antar annotator,
    - melihat apakah final score advanced aggregation masuk akal,
    - digunakan untuk analisis laporan.
    """

    high_disagreement = aggregated_df.sort_values(
        by="std_score",
        ascending=False
    ).head(top_n)

    for _, row in high_disagreement.iterrows():
        qid = row["query_id"]
        did = row["doc_id"]

        group = raw_df[
            (raw_df["query_id"] == qid) &
            (raw_df["doc_id"].astype(str) == str(did))
        ]

        print("=" * 80)
        print(f"Query ID             : {qid}")
        print(f"Doc ID               : {did}")
        print(f"Aggregated score     : {row['score']}")
        print(f"Number of judgements : {row['num_judgements']}")
        print(f"Std score            : {row['std_score']:.2f}")
        print("\nRaw judgements:")
        display(group)

In [79]:
inspect_high_disagreement_cases(
    raw_df=df,
    aggregated_df=agg_weighted,
    top_n=5
)

Query ID             : 9
Doc ID               : d9_3
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
198,9,d9_3,0,0.67,User_2,43300
199,9,d9_3,0,0.96,User_5,17720
200,9,d9_3,2,0.64,User_3,61968


Query ID             : 6
Doc ID               : d6_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
138,6,d6_7,0,0.97,User_1,89656
139,6,d6_7,2,0.85,User_4,23359
140,6,d6_7,2,0.55,User_2,72910


Query ID             : 4
Doc ID               : d4_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
90,4,d4_7,0,0.93,User_5,55279
91,4,d4_7,2,0.82,User_1,51363
92,4,d4_7,2,0.87,User_2,50752


Query ID             : 9
Doc ID               : d9_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
195,9,d9_2,3,0.67,User_5,56548
196,9,d9_2,3,0.56,User_1,51132
197,9,d9_2,1,0.59,User_0,75483


Query ID             : 6
Doc ID               : d6_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.82

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
123,6,d6_2,1,0.95,User_3,61692
124,6,d6_2,3,0.67,User_2,45065
125,6,d6_2,2,0.95,User_1,18826


## 1.6 Simpan Semua

In [ ]:
save_qrels(agg_maj, DEST + "fira_aggregated.qrels")
save_qrels(agg_weighted, DEST + "fira_aggregated_confidence_weighted.qrels")
save_qrels(agg_reliability, DEST + "fira_aggregated_annotator_reliability.qrels")

# **Part 2: BERT Cross-Encoder Re-Ranking**

In [ ]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "How to make a good cappuccino?"
passages = ["Passage 1 ...", "Passage 2 ...", "Passage 3 ..."]
ranked_idx, scores = reranker.re_rank(query, passages)
print("Ranked order:", ranked_idx)

# **Part 3: Extractive QA**

In [ ]:
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

result = qa_pipeline(question=query, context=passages[ranked_idx[0]])
print(result)

## Evaluasi
# TODO: Implement evaluasi MRR@10, NDCG@10, dll.